# X-CLIP: ActionCLIP Style Prompting on UCF-101

This notebook evaluates **X-CLIP** (Microsoft) using **ActionCLIP Prompting** on the UCF-101 dataset.

Results are streamed to a CSV file, including inference duration per clip.

In [ ]:
from helpers import (
    precompute_action_clip, 
    stack_action_clip, 
    run_action_clip_inference, 
    get_clip_generator
)
from datasets import load_dataset
import torch
import os
import json
import csv
import time
from tqdm import tqdm

# 1. Load Prompts & Precompute Embeddings
PROMPT_PATH = "ucf101_actionclip_prompts.json"
dictionary = precompute_action_clip(PROMPT_PATH)
matrices = stack_action_clip(dictionary)

# 2. Load Dataset (UCF-101 via HuggingFace)
dataset = load_dataset("flwrlabs/ucf101")
dataset = dataset['test']


Precomputing X-CLIP embeddings for 101 classes (ActionCLIP Style)...


Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/79 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

In [6]:
# 3. Run Inference & Stream to CSV
os.makedirs("results", exist_ok=True)
RES_PATH = "results/ucf101_regular_xclip_results.csv"
MAX_CLIPS = 10000

with open(RES_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'clip_id', 'label_name', 'predicted_class',
        'final_score', 'num_frames','duration'
    ])

    start_time = time.time()

    for i, row in enumerate(get_clip_generator(dataset, max_clips=MAX_CLIPS)):
        clip_frames = row["images"]
        num_frames  = len(clip_frames)

        if not clip_frames:
            print(f"Skipping {row['clip_id']}: No frames.")
            continue

        clip_label = row['label_name']

        inference_start = time.time()
        # Use run_action_clip_inference for regular prompting
        clip_out = run_action_clip_inference(clip_frames, matrices)
        duration = time.time() - inference_start

        writer.writerow([
            row['clip_id'],
            clip_label,
            clip_out['predicted_class'],
            clip_out['final_score'],
            num_frames,
            duration
        ])

        print(f"[{i+1}] Predicted: {clip_out['predicted_class']:20s} | Actual: {clip_label:20s}")

    end_time = time.time()
    print(f"\nTotal time: {end_time - start_time:.2f} seconds")
    print(f"Results streamed to {RES_PATH}")

[1] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[2] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[3] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[4] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[5] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[6] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[7] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[8] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[9] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[10] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[11] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[12] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[13] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[14] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMakeup      
[15] Predicted: ApplyEyeMakeup       | Actual: ApplyEyeMa

In [7]:
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
import pandas as pd

RES_PATH = "results/ucf101_regular_xclip_results.csv"
df = pd.read_csv(RES_PATH)

accuracy_count = (df['label_name'] == df['predicted_class']).sum()
total_count    = len(df)
accuracy       = accuracy_count / total_count * 100

print(f"📊 Final Accuracy: {accuracy_count}/{total_count} = {accuracy:.4f}%")


print(f"⏱  Average Latency per frame: {(df['duration']/ 8).mean():.4f} seconds")

# --- Confusion Matrix ---
confusion_matrix = pd.crosstab(
    df['label_name'],
    df['predicted_class'],
    rownames=['Actual'],
    colnames=['Predicted']
)

fig = px.imshow(
    confusion_matrix,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=confusion_matrix.columns,
    y=confusion_matrix.index,
    color_continuous_scale='Blues',
    aspect="auto"
)

fig.update_layout(
    title='SP-XCLIP: Interactive UCF-101 Confusion Matrix',
    xaxis_nticks=101,
    yaxis_nticks=101,
    width=1200,
    height=1000
)

pio.renderers.default = "browser"
fig.write_html("results/ucf101_regular_xclip_confusion_matrix.html")
fig.show()

📊 Final Accuracy: 3165/3783 = 83.6638%
⏱  Average Latency per frame: 0.0225 seconds
